# L4M R&D: parameter sets per use case

**Goal:** identify a set of optimised welding parameters for each use case (Fusion welding, Pharma welding, Battery cooling welding).
**Reference flow:** 1 set of optimised parameters for one use case.
**Scenarios per use case:**
- *SoA R&D*: all experiments run physically in the lab with the standard laser.
- *L4M-enabled R&D*: fewer physical experiments, plus digital lab runs, with the joining done by Dynamic Beam Shaping (DBS).

Data come from `databases/L4M_database.bw2package` (`L4M system`). See `Diagrams/L4M_RD_parameter_sets.md` for the process flow.
The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [2]:
from pathlib import Path

import bw2data as bd
import numpy as np
import pandas as pd
from IPython.display import display

from l4m_lca import plots
from l4m_lca.constants import EU27
from l4m_lca.impacts import BackgroundImpacts, bw2calc_scores, foreground_flows
from l4m_lca.project import (DEFAULT_INDICATORS, category, check_links, ef31_methods, ensure_package,
                             method_units, open_project, short)
from l4m_lca.rd import build_rd_database, make_round, process_label, sankey_links, standard_welds_left
from l4m_lca.regional import electricity_markets, system_regional

## Parameters

In [3]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"
PACKAGE_DIR = Path("databases")

L4M_NAME = "L4M system"
METALLURGIC_NAME = "Mettalurgic test (CIVAN)"      # spelling as stored in the packages
CT_SCAN_NAME = "CT-scan test"
RD_NAME = "L4M_RD_parameter_sets"
REIMPORT_L4M = False   # True: delete and import L4M system again (discards edits made in the project)

INDICATORS = DEFAULT_INDICATORS

DBS_ENERGY_FACTOR = 0.8   # DBS uses 20% less electricity than the standard laser (all electricity of the joining steps)
OPTIMIZATION_KWH = 1.0    # PLACEHOLDER: electricity of one optimization run (kWh)

# PLACEHOLDERS (read from the process diagram): per round, the number of coupon welds, prototype welds,
# digital lab runs, optimization runs and verifications (cutting, polishing, etching and CT-scan of the joined welds).
# "joining" is the technology of the coupon and prototype joining.
def scenarios(prototype_soa, prototype_l4m):
    return {
        "SoA R&D": {"joining": "Standard laser", "rounds": {
            "Coupon round 1":    make_round(coupon=150),
            "Coupon round 2":    make_round(coupon=150),
            "Prototype round 1": make_round(prototype=prototype_soa)}},
        "L4M-enabled R&D": {"joining": "Dynamic Beam Shaping", "rounds": {
            "Coupon round 1":    make_round(coupon=100, digital=1),
            "Coupon round 2":    make_round(coupon=100, optimization=4),
            "Prototype round 1": make_round(prototype=prototype_l4m)}},
    }

USE_CASE_NAMES = ["Fusion welding", "Pharma welding", "Battery cooling welding"]
RD_SCENARIOS = {uc: scenarios(prototype_soa=50, prototype_l4m=75) for uc in USE_CASE_NAMES}

In [4]:
# Saved processes: label -> (database, code). The comment gives the reference flow of each process:
# the amount of its reference product that it produces. All its input amounts refer to this flow.
L4M_PROCESSES = {
    "001_Physical lab (battery cooling pre-processing + welding)": "a7cf0ea95c2a499aa8cf9eb0ba72142c",  # 1 "1 set of optimised parameters"
    "001_Physical_Cutting":       "97a5ed8baec4406f879d5cff60e87345_copy1",  # 500 "Cut coupon"
    "001_Physical_Laser welding": "5a81aade45924c56854be233364a0435_copy1",  # 250 "Weld coupon"
    "001_Pyshical_Cleaning":      "fbbf6041c24c4e28b6c23a3ab0eebf0a_copy1",  # 500 "Clean coupon"
    "Battery cooling welding":    "68f4a5e21189435ba9b0a376e068edbc",        # 1 "Battery cooling plate"
    "CT-scan X-ray":              "93d9467878cb43ba9798a256e98512f5_copy1",  # 1 "Tested coupon" (location GB)
    "Digital lab processes":      "d273f34fee714ae0b25dbfb39576ab5b",        # 1 "Digital lab processes"
    "Fusion welding":             "68f4a5e21189435ba9b0a376e068edbc_copy2",  # 1 "Pipe welding"
    "Pharma welding":             "68f4a5e21189435ba9b0a376e068edbc_copy1",  # 1 "Freeze dryer shelf"
    "Platform":                   "50a84844a3de4677926d511e171e31d5",        # 1 "Platform"
    "Prediction":                 "8fbb7937f5224de9b836ae54858b6fe1",        # 1 "Prediction"
}
METALLURGIC_PROCESSES = {
    "01_Cutting":       "97a5ed8baec4406f879d5cff60e87345",  # 132 "Cut coupon"
    "02_Cleaning":      "fbbf6041c24c4e28b6c23a3ab0eebf0a",  # 1 "Clean coupon"
    "03_Laser welding": "5a81aade45924c56854be233364a0435",  # 1 "Weld coupon"
    "04_Cutting":       "7faf722864b94ed797cdd406d22120db",  # 3 "Weld cross-sections"
    "05_Hot-mounting":  "06923c5d5a4d416d87c1702173dc648a",  # 1 "Mounted sample"
    "06_Grinding":      "f00138e49c5843caa7a1e7d8f33f5986",  # 6 "Grinded sample"
    "07_Polishing":     "53c6381979c942e79ed1f043a8b184ed",  # 6 "Polished sample"
    "08_Etching":       "92039648c9a64364ab22f228c0f4b44d",  # 18 "Etched sample"
    "09_Test":          "32bb94122a4542ad8af12fb8ea34a26e",  # 18 "Tested sample"
}
CT_SCAN_PROCESSES = {
    "01_Cutting":       "97a5ed8baec4406f879d5cff60e87345",  # 132 "Cut coupon"
    "02_Cleaning":      "fbbf6041c24c4e28b6c23a3ab0eebf0a",  # 1 "Clean coupon"
    "03_Laser welding": "5a81aade45924c56854be233364a0435",  # 1 "Weld coupon"
    "04_Cutting":       "7faf722864b94ed797cdd406d22120db",  # 1 "Weld cross-sections"
    "05_CT-scan":       "93d9467878cb43ba9798a256e98512f5",  # 1 "Tested coupon"
}

# Plot groups: process type -> {process label: colour} (legend order)
PROCESS_TYPES = {
    "Pre-processing":  {"Raw material sourcing": "#0f3f7a", "Cutting": "#2a78d6", "Cleaning": "#8fb8ee"},
    "Joining":         {"Coupon joining": "#eb6834", "Prototype joining": "#f4a582"},
    "Post-processing": {"Cutting (weld cross-sections)": "#4b3bb5", "Polishing": "#8d80e0", "Etching": "#b0a6ec"},
    "Testing":         {"CT-scan X-ray": "#1baf7a"},
    "Digital lab":     {"Platform": "#eda100", "Prediction": "#e87ba4", "Optimization": "#f6cf6b"},
}

# Regional analysis: EU27 plus GB (the physical lab is modelled with GB electricity). Disabled by default.
SHOW_SANKEY = False
SHOW_REGIONAL = False
REGIONAL_COUNTRIES = {**EU27, "GB": "GBR"}

## Project and imported databases

In [5]:
ei = open_project(PROJECT_NAME, ECOINVENT_DATABASE_NAME)
# L4M system links to the other two databases, so they come first
ensure_package(CT_SCAN_NAME, PACKAGE_DIR / "CT-scan test.bw2package")
ensure_package(METALLURGIC_NAME, PACKAGE_DIR / "Metallurgic test (CIVAN).bw2package")
ensure_package(L4M_NAME, PACKAGE_DIR / "L4M_database.bw2package", force=REIMPORT_L4M)
check_links([L4M_NAME, METALLURGIC_NAME, CT_SCAN_NAME])

methods = ef31_methods(INDICATORS)
units = method_units(methods)
indicators = [category(m) for m in methods]
DEFAULT_INDICATOR = next((i for i in indicators if i.startswith("climate change")), indicators[0])

CT-scan test: 5 activities already in the project, import skipped.
Mettalurgic test (CIVAN): 9 activities already in the project, import skipped.
L4M system: 11 activities already in the project, import skipped.


In [6]:
l4m = {label: bd.Database(L4M_NAME).get(code) for label, code in L4M_PROCESSES.items()}
metallurgic = {label: bd.Database(METALLURGIC_NAME).get(code) for label, code in METALLURGIC_PROCESSES.items()}
ct_scan = {label: bd.Database(CT_SCAN_NAME).get(code) for label, code in CT_SCAN_PROCESSES.items()}

# Reference flow of each saved process, to check the comments above
pd.DataFrame(
    [{"Database": a["database"], "Process": label, "Reference product": a.get("reference product"),
      "Location": a.get("location"), "Unit": a["unit"],
      "Amount": next(e["amount"] for e in a.exchanges() if e["type"] == "production")}
     for procs in (l4m, metallurgic, ct_scan) for label, a in procs.items()])

,Database,Process,Reference product,Location,Unit,Amount
0,L4M system,001_Physical lab (battery cooling pre-processi...,1 set of optimised parameters,GLO,unit,1.0
1,L4M system,001_Physical_Cutting,Cut coupon,GLO,unit,500.0
2,L4M system,001_Physical_Laser welding,Weld coupon,GLO,unit,250.0
3,L4M system,001_Pyshical_Cleaning,Clean coupon,GLO,unit,500.0
4,L4M system,Battery cooling welding,Battery cooling plate,GLO,unit,1.0
5,L4M system,CT-scan X-ray,Tested coupon,GB,unit,1.0
6,L4M system,Digital lab processes,Digital lab processes,GLO,unit,1.0
7,L4M system,Fusion welding,Pipe welding,GLO,unit,1.0
8,L4M system,Pharma welding,Freeze dryer shelf,GLO,unit,1.0
9,L4M system,Platform,Platform,GLO,unit,1.0


## R&D product systems

One product system per use case and scenario. The set of parameters is identified in rounds (coupon round 1, coupon round 2, prototype round 1).
In the L4M-enabled scenario, the digital platform investigates parameter sets, and an optimization run uses the results of the successful tests before the next physical round.
Every physical round is verified by cutting, polishing, etching and CT-scanning the joined welds.

In [7]:
use_cases = {uc: l4m[uc] for uc in USE_CASE_NAMES}
systems = build_rd_database(RD_NAME, ECOINVENT_DATABASE_NAME, l4m, metallurgic, ct_scan, use_cases,
                            RD_SCENARIOS, DBS_ENERGY_FACTOR, OPTIMIZATION_KWH)
list(systems)

100%|██████████| 38/38 [00:00<00:00, 6202.18it/s]

11:25:21+0100 [info     ] Vacuuming database            


[('Fusion welding', 'SoA R&D'),
 ('Fusion welding', 'L4M-enabled R&D'),
 ('Pharma welding', 'SoA R&D'),
 ('Pharma welding', 'L4M-enabled R&D'),
 ('Battery cooling welding', 'SoA R&D'),
 ('Battery cooling welding', 'L4M-enabled R&D')]

## Impact calculation

Each product system is walked through the foreground processes to the background inputs (ecoinvent markets, biosphere flows).
The impact of 1 unit of each background input is calculated one time, then scaled by the amount for 1 set of parameters.
The next cell checks the totals against a regular `bw2calc` LCA.

In [8]:
FOREGROUND = {L4M_NAME, METALLURGIC_NAME, CT_SCAN_NAME, RD_NAME}
impacts = BackgroundImpacts(methods, next(iter(ei)))
flows = {k: foreground_flows(act, FOREGROUND) for k, act in systems.items()}
contribs = {k: impacts.by_flow(f) for k, f in flows.items()}
totals = {k: sum(v for _, v in c) for k, c in contribs.items()}

for (uc, sc), f in flows.items():
    if RD_SCENARIOS[uc][sc]["joining"] == "Dynamic Beam Shaping":
        left = standard_welds_left(f, use_cases)
        assert not left, f"Standard-laser weld left in {uc} | {sc}: {left}"
for k, act in systems.items():
    assert np.allclose(totals[k], bw2calc_scores(act, methods), rtol=1e-6), f"Walk does not reproduce bw2calc for {k}"
print(f"Walk matches bw2calc for all {len(systems)} product systems.")

pd.DataFrame({k: dict(zip(indicators, v)) for k, v in totals.items()}).T.rename_axis(["Use case", "Scenario"])

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)
/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 7.65e+12)
  warnings.warn(msg, UmfpackWarning)


Walk matches bw2calc for all 6 product systems.


acidification  \
Use case                Scenario                         
Fusion welding          SoA R&D               0.635191   
                        L4M-enabled R&D       0.464661   
Pharma welding          SoA R&D               0.612894   
                        L4M-enabled R&D       0.436722   
Battery cooling welding SoA R&D               0.600876   
                        L4M-enabled R&D       0.421332   

                                         eutrophication: terrestrial  \
Use case                Scenario                                       
Fusion welding          SoA R&D                             1.177155   
                        L4M-enabled R&D                     0.850571   
Pharma welding          SoA R&D                             1.143039   
                        L4M-enabled R&D                     0.807809   
Battery cooling welding SoA R&D                             1.124634   
                        L4M-enabled R&D                     0.784230   

                                         human toxicity: carcinogenic  \
Use case                Scenario                                        
Fusion welding          SoA R&D                          6.110610e-08   
                        L4M-enabled R&D                  4.218174e-08   
Pharma welding          SoA R&D                          6.034257e-08   
                        L4M-enabled R&D                  4.122419e-08   
Battery cooling welding SoA R&D                          5.993009e-08   
                        L4M-enabled R&D                  4.069540e-08   

                                         human toxicity: carcinogenic, inorganics  \
Use case                Scenario                                                    
Fusion welding          SoA R&D                                      1.894942e-08   
                        L4M-enabled R&D                              1.351267e-08   
Pharma welding          SoA R&D                                      1.846819e-08   
                        L4M-enabled R&D                              1.290955e-08   
Battery cooling welding SoA R&D                                      1.820866e-08   
                        L4M-enabled R&D                              1.257713e-08   

                                         human toxicity: carcinogenic, organics  \
Use case                Scenario                                                  
Fusion welding          SoA R&D                                    4.215668e-08   
                        L4M-enabled R&D                            2.866907e-08   
Pharma welding          SoA R&D                                    4.187438e-08   
                        L4M-enabled R&D                            2.831464e-08   
Battery cooling welding SoA R&D                                    4.172143e-08   
                        L4M-enabled R&D                            2.811827e-08   

                                         particulate matter formation  \
Use case                Scenario                                        
Fusion welding          SoA R&D                              0.000015   
                        L4M-enabled R&D                      0.000010   
Pharma welding          SoA R&D                              0.000015   
                        L4M-enabled R&D                      0.000010   
Battery cooling welding SoA R&D                              0.000015   
                        L4M-enabled R&D                      0.000010   

                                         photochemical oxidant formation: human health  \
Use case                Scenario                                                         
Fusion welding          SoA R&D                                               0.348074   
                        L4M-enabled R&D                                       0.252540   
Pharma welding          SoA R&D                                               0.337383   
                        L4M-enabl

## Stacked bar: SoA R&D vs L4M-enabled R&D

Impact of 1 set of optimised parameters, split by the process that causes it directly. The legend groups the processes into pre-processing, joining, post-processing, testing and digital lab.

In [9]:
weld_names = {p["name"] for p in use_cases.values()}
names = {k: bd.get_activity(k)["name"] for c in contribs.values() for f, _ in c for k in f.path}

def stack_table(indicator):
    """Impact per set of parameters caused directly by each process (rows: systems, columns: process labels)."""
    i = indicators.index(indicator)
    out = {}
    for k, c in contribs.items():
        row = {}
        for flow, v in c:
            label = process_label(names[flow.path[-1]], weld_names)
            row[label] = row.get(label, 0.0) + v[i]
        out[k] = row
    return pd.DataFrame(out).T.fillna(0.0)

display(plots.grouped_stacked_bars(stack_table, indicators, DEFAULT_INDICATOR, units, PROCESS_TYPES,
                                   title=lambda ind: f"{short(ind)} per set of optimised parameters"))

## Sankey: where the impact flows

Background inputs (left) feed the foreground processes, which pass their cumulative impact on to the set of parameters (right). Set `SHOW_SANKEY = True` to show it.

In [10]:
if SHOW_SANKEY:
    db_tag = {L4M_NAME: "L4M", METALLURGIC_NAME: "Metallurgic", CT_SCAN_NAME: "CT-scan", RD_NAME: ""}

    def node_label(key):
        act = bd.get_activity(key)
        tag = db_tag.get(act["database"], "")
        return f"{act['name']} ({tag})" if tag else act["name"]

    display(plots.sankey(
        links_of=lambda k, ind: sankey_links(contribs[k], indicators.index(ind), node_label),
        systems={f"{uc} | {sc}": (uc, sc) for uc, sc in systems},
        indicators=indicators, default=DEFAULT_INDICATOR, units=units,
        totals=lambda k, ind: totals[k][indicators.index(ind)],
        root_name=lambda k: systems[k]["name"]))

## Regional variation: EU electricity markets

Every medium-voltage electricity input (the RER market group and the GB markets of the physical lab) is replaced by the market of one country. Left: SoA R&D, right: L4M-enabled R&D. Set `SHOW_REGIONAL = True` to show it.

In [11]:
if SHOW_REGIONAL:
    markets = electricity_markets(ei, REGIONAL_COUNTRIES)
    reg = system_regional(systems, flows, impacts, markets, baseline="As modelled")
    display(plots.paired_maps(
        value=lambda uc, sc, c, ind: reg[(uc, sc), c][indicators.index(ind)],
        use_cases=list(RD_SCENARIOS), scenarios=["SoA R&D", "L4M-enabled R&D"],
        indicators=indicators, default=DEFAULT_INDICATOR, units=units, countries=REGIONAL_COUNTRIES,
        baseline="As modelled", title=lambda uc, ind: f"{uc}: {short(ind)} per set of parameters"))